# Modelo Multimodal: MTDE-Net (Baseline vs. Optimizado)

Este cuaderno implementa de manera secuencial el entrenamiento y comparación de **MTDE-Net**:
1. **Fase A (Baseline):** Entrenamiento con la configuración inicial por defecto (sin sintonizar).
2. **Fase B (Optimizado):** Entrenamiento con la configuración de hiperparámetros campeona encontrada mediante optimización bayesiana con Optuna y semilla fija.

Al final del cuaderno, se genera de manera automática una **tabla comparativa de métricas** lista para incluir en el reporte final de tu tesis.

In [1]:
import sys
import random
from pathlib import Path

# Añadir el directorio raíz al path de Python para permitir importaciones correctas
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

from src.models.mtde_net import MTDE_Net
from src.loaders.mtde_net_loader import MultimodalThermalDataset
from src.utils import SqrtScaledMSELoss, eval_mtde_net_metrics, split_by_sequence

### 1. Semilla Global y Reproducibilidad

In [2]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

### 2. Carga del Dataset y Partición Estricta (Compartida)

In [3]:
dev = "cuda" if torch.cuda.is_available() else "cpu"
time_scale = 30.0

train_ds_full = MultimodalThermalDataset(
    metadata_csv="../processed_data/metadata.csv", 
    is_train=True, 
    min_time_s=0.0,
    time_scale=time_scale
)
val_ds_full = MultimodalThermalDataset(
    metadata_csv="../processed_data/metadata.csv", 
    is_train=False, 
    min_time_s=0.0,
    time_scale=time_scale
)

# Forzar el mapeo de ruta para entorno notebook
train_ds_full.root = Path("../processed_data")
val_ds_full.root = Path("../processed_data")

t_idx, v_idx = split_by_sequence(train_ds_full.df)

print(f"Dataset cargado exitosamente. Particiones: {len(t_idx)} train / {len(v_idx)} val")

Dataset cargado exitosamente. Particiones: 1111 train / 329 val


--- 
## FASE A: Entrenamiento del Modelo Baseline (Sin Tunear)

Esta sección entrena el modelo con los hiperparámetros iniciales por defecto.

In [4]:
# 1. Configuración de Hiperparámetros Baseline
BASELINE_CONFIG = {
    "epochs": 120,
    "patience": 15,
    "min_delta": 1.0,
    "batch_size": 16,
    "lr": 0.0005,
    "weight_decay": 0.0005,
    "dropout": 0.2
}

set_seed(42)

# 2. Dataloaders específicos para Baseline
train_loader_base = DataLoader(Subset(train_ds_full, t_idx), batch_size=BASELINE_CONFIG["batch_size"], shuffle=True)
val_loader_base = DataLoader(Subset(val_ds_full, v_idx), batch_size=BASELINE_CONFIG["batch_size"])
train_eval_loader_base = DataLoader(Subset(train_ds_full, t_idx), batch_size=BASELINE_CONFIG["batch_size"])

# 3. Inicializar Modelo Baseline
model_base = MTDE_Net(dropout=BASELINE_CONFIG["dropout"]).to(dev)
crit = SqrtScaledMSELoss(scale=None)
opt = torch.optim.AdamW(model_base.parameters(), lr=BASELINE_CONFIG["lr"], weight_decay=BASELINE_CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[30, 60, 80], gamma=0.8)

print(f"Iniciando entrenamiento Baseline... Parámetros: {sum(p.numel() for p in model_base.parameters()):,}")

# 4. Ciclo de Entrenamiento
best_mae_base = float("inf")
no_imp = 0

for ep in range(1, BASELINE_CONFIG["epochs"] + 1):
    model_base.train()
    train_loss, n = 0.0, 0
    for x_img, x_tab, y in train_loader_base:
        x_img, x_tab, y = x_img.to(dev), x_tab.to(dev), y.to(dev)
        opt.zero_grad(set_to_none=True)
        loss = crit(model_base(x_img, x_tab), y)
        loss.backward()
        opt.step()
        train_loss += loss.item() * x_img.size(0)
        n += x_img.size(0)

    scheduler.step()
    
    # Evaluar métricas unificadas en segundos reales
    val_m = eval_mtde_net_metrics(model_base, val_loader_base, dev, scale=time_scale)
    train_m = eval_mtde_net_metrics(model_base, train_eval_loader_base, dev, scale=time_scale)
    
    v_mae = val_m["mae"]
    is_best = v_mae < best_mae_base - BASELINE_CONFIG["min_delta"]
    if is_best: 
        best_mae_base, no_imp = v_mae, 0
        torch.save(model_base.state_dict(), "../MTDE_Net_baseline.pt")
        baseline_metrics = val_m # Guardar para comparación final
    else: 
        no_imp += 1
        
    print(f"Ep {ep:03d} | Loss: {train_loss/n:.4f} | TrMAE: {train_m['mae']:5.2f}s | ValMAE: {v_mae:5.2f}s | "
          f"RMSE: {val_m['rmse']:5.2f}s | R2: {val_m['r2']:.4f} | MAPE: {val_m['mape']:5.2f}% | "
          f"Acc60: {val_m['acc60']:.2f}% | Acc120: {val_m['acc120']:.2f}% {'*' if is_best else ''}")
    
    if no_imp >= BASELINE_CONFIG["patience"]:
        print(f"Early stop alcanzado. Mejor Val MAE Baseline: {best_mae_base:.2f}s")
        break

Iniciando entrenamiento Baseline... Parámetros: 1,325,421
Ep 001 | Loss: 1.4688 | TrMAE: 95.45s | ValMAE: 110.14s | RMSE: 149.91s | R2: 0.1732 | MAPE: 64.73% | Acc60: 42.86% | Acc120: 65.65% *
Ep 002 | Loss: 0.7081 | TrMAE: 69.03s | ValMAE: 84.57s | RMSE: 118.41s | R2: 0.4842 | MAPE: 52.06% | Acc60: 52.28% | Acc120: 77.20% *
Ep 003 | Loss: 0.4002 | TrMAE: 70.52s | ValMAE: 75.47s | RMSE: 101.56s | R2: 0.6205 | MAPE: 52.32% | Acc60: 54.41% | Acc120: 81.46% *
Ep 004 | Loss: 0.3622 | TrMAE: 50.99s | ValMAE: 54.21s | RMSE: 79.09s | R2: 0.7699 | MAPE: 38.39% | Acc60: 72.95% | Acc120: 86.63% *
Ep 005 | Loss: 0.3448 | TrMAE: 58.05s | ValMAE: 67.78s | RMSE: 96.62s | R2: 0.6566 | MAPE: 48.99% | Acc60: 61.40% | Acc120: 87.84% 
Ep 006 | Loss: 0.3224 | TrMAE: 38.22s | ValMAE: 43.22s | RMSE: 58.33s | R2: 0.8748 | MAPE: 37.26% | Acc60: 72.34% | Acc120: 94.22% *
Ep 007 | Loss: 0.2038 | TrMAE: 32.11s | ValMAE: 36.83s | RMSE: 49.72s | R2: 0.9091 | MAPE: 47.02% | Acc60: 78.42% | Acc120: 97.26% *
Ep 008 |

--- 
## FASE B: Entrenamiento del Modelo Optimizado (Optuna Champion)

Esta sección entrena el modelo con los hiperparámetros óptimos encontrados en tu estudio determinista con Optuna.

In [5]:
# 1. Configuración de Hiperparámetros Optimizados
OPTIMIZED_CONFIG = {
    "epochs": 120,
    "patience": 15,
    "min_delta": 1.0,
    'lr': 0.0007311180257053372, 
    'weight_decay': 0.0006371219710483732, 
    'batch_size': 32, 
    'dropout': 0.3359529946600127
}

set_seed(42)

# 2. Dataloaders específicos para Optimizado
train_loader_opt = DataLoader(Subset(train_ds_full, t_idx), batch_size=OPTIMIZED_CONFIG["batch_size"], shuffle=True)
val_loader_opt = DataLoader(Subset(val_ds_full, v_idx), batch_size=OPTIMIZED_CONFIG["batch_size"])
train_eval_loader_opt = DataLoader(Subset(train_ds_full, t_idx), batch_size=OPTIMIZED_CONFIG["batch_size"])

# 3. Inicializar Modelo Optimizado
model_opt = MTDE_Net(dropout=OPTIMIZED_CONFIG["dropout"]).to(dev)
crit = SqrtScaledMSELoss(scale=None)
opt = torch.optim.AdamW(model_opt.parameters(), lr=OPTIMIZED_CONFIG["lr"], weight_decay=OPTIMIZED_CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[30, 60, 80], gamma=0.8)

print(f"Iniciando entrenamiento Optimizado... Parámetros: {sum(p.numel() for p in model_opt.parameters()):,}")

# 4. Ciclo de Entrenamiento
best_mae_opt = float("inf")
no_imp = 0

for ep in range(1, OPTIMIZED_CONFIG["epochs"] + 1):
    model_opt.train()
    train_loss, n = 0.0, 0
    for x_img, x_tab, y in train_loader_opt:
        x_img, x_tab, y = x_img.to(dev), x_tab.to(dev), y.to(dev)
        opt.zero_grad(set_to_none=True)
        loss = crit(model_opt(x_img, x_tab), y)
        loss.backward()
        opt.step()
        train_loss += loss.item() * x_img.size(0)
        n += x_img.size(0)

    scheduler.step()
    
    # Evaluar métricas unificadas en segundos reales
    val_m = eval_mtde_net_metrics(model_opt, val_loader_opt, dev, scale=time_scale)
    train_m = eval_mtde_net_metrics(model_opt, train_eval_loader_opt, dev, scale=time_scale)
    
    v_mae = val_m["mae"]
    is_best = v_mae < best_mae_opt - OPTIMIZED_CONFIG["min_delta"]
    if is_best: 
        best_mae_opt, no_imp = v_mae, 0
        torch.save(model_opt.state_dict(), "../MTDE_Net_best.pt")
        optimized_metrics = val_m # Guardar para comparación final
    else: 
        no_imp += 1
        
    print(f"Ep {ep:03d} | Loss: {train_loss/n:.4f} | TrMAE: {train_m['mae']:5.2f}s | ValMAE: {v_mae:5.2f}s | "
          f"RMSE: {val_m['rmse']:5.2f}s | R2: {val_m['r2']:.4f} | MAPE: {val_m['mape']:5.2f}% | "
          f"Acc60: {val_m['acc60']:.2f}% | Acc120: {val_m['acc120']:.2f}% {'*' if is_best else ''}")
    
    if no_imp >= OPTIMIZED_CONFIG["patience"]:
        print(f"Early stop alcanzado. Mejor Val MAE Optimizado: {best_mae_opt:.2f}s")
        break

Iniciando entrenamiento Optimizado... Parámetros: 1,325,421
Ep 001 | Loss: 1.6135 | TrMAE: 78.19s | ValMAE: 69.93s | RMSE: 101.89s | R2: 0.6181 | MAPE: 63.65% | Acc60: 60.79% | Acc120: 85.11% *
Ep 002 | Loss: 0.7954 | TrMAE: 97.19s | ValMAE: 108.24s | RMSE: 145.30s | R2: 0.2233 | MAPE: 65.03% | Acc60: 39.51% | Acc120: 70.52% 
Ep 003 | Loss: 0.5242 | TrMAE: 97.47s | ValMAE: 92.21s | RMSE: 121.64s | R2: 0.4556 | MAPE: 62.11% | Acc60: 41.34% | Acc120: 76.90% 
Ep 004 | Loss: 0.3949 | TrMAE: 66.00s | ValMAE: 69.54s | RMSE: 90.67s | R2: 0.6976 | MAPE: 51.65% | Acc60: 51.67% | Acc120: 85.11% 
Ep 005 | Loss: 0.3076 | TrMAE: 61.85s | ValMAE: 76.74s | RMSE: 113.02s | R2: 0.5301 | MAPE: 45.62% | Acc60: 56.84% | Acc120: 78.72% 
Ep 006 | Loss: 0.2330 | TrMAE: 49.05s | ValMAE: 53.37s | RMSE: 72.60s | R2: 0.8061 | MAPE: 41.66% | Acc60: 68.69% | Acc120: 91.79% *
Ep 007 | Loss: 0.1834 | TrMAE: 36.27s | ValMAE: 40.48s | RMSE: 55.38s | R2: 0.8872 | MAPE: 37.45% | Acc60: 78.42% | Acc120: 94.53% *
Ep 008 |

--- 
## 3. Tabla Comparativa de Resultados Finales

Esta celda autogenera una tabla markdown limpia comparando ambos escenarios listos para exportar.

In [6]:
comparative_data = {
    "Métrica": ["MAE (Error Absoluto Medio)", "RMSE (Error Cuadrático Medio)", "R² (Coeficiente de Det.)", "MAPE (Error Porcentual)", "Acc@60s (Exactitud 1 min)", "Acc@120s (Exactitud 2 min)"],
    "Baseline (Default)": [
        f"{baseline_metrics['mae']:.2f} s",
        f"{baseline_metrics['rmse']:.2f} s",
        f"{baseline_metrics['r2']:.4f}",
        f"{baseline_metrics['mape']:.2f} %",
        f"{baseline_metrics['acc60']:.2f} %",
        f"{baseline_metrics['acc120']:.2f} %"
    ],
    "Optimizado (Optuna)": [
        f"{optimized_metrics['mae']:.2f} s",
        f"{optimized_metrics['rmse']:.2f} s",
        f"{optimized_metrics['r2']:.4f}",
        f"{optimized_metrics['mape']:.2f} %",
        f"{optimized_metrics['acc60']:.2f} %",
        f"{optimized_metrics['acc120']:.2f} %"
    ]
}

df_comparison = pd.DataFrame(comparative_data)
from IPython.display import display, Markdown
display(Markdown("### Tabla Comparativa de MTDE-Net para Tesis"))
display(df_comparison)

### Tabla Comparativa de MTDE-Net para Tesis

,Métrica,Baseline (Default),Optimizado (Optuna)
0,MAE (Error Absoluto Medio),22.39 s,22.26 s
1,RMSE (Error Cuadrático Medio),31.10 s,30.14 s
2,R² (Coeficiente de Det.),0.9644,0.9666
3,MAPE (Error Porcentual),25.33 %,20.82 %
4,Acc@60s (Exactitud 1 min),92.71 %,93.62 %
5,Acc@120s (Exactitud 2 min),99.70 %,100.00 %
